In [3]:
import pandas as pd

df = pd.read_csv('../data/Resume.csv')

In [4]:
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [5]:
df.shape

(2484, 4)

In [6]:
df['Category'].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [7]:
df['Resume_str'][0]

"         HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss pr

In [8]:
import re

def clean_text(text):
    text = text.lower()                          # lowercase everything
    text = re.sub(r'\n', ' ', text)               # replace newlines with space
    text = re.sub(r'\s+', ' ', text)              # collapse multiple spaces into one
    text = text.strip()                           # remove leading/trailing spaces
    return text

df['cleaned_resume'] = df['Resume_str'].apply(clean_text)
print(df['cleaned_resume'][0])

hr administrator/marketing associate hr administrator summary dedicated customer service manager with 15+ years of experience in hospitality and customer service management. respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service. highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multi-tasker client relations specialist accomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification accomplished trainer for cross server hospitality systems such as hilton onq , micros opera pms , fidelio opera reservation system (ors) , holidex completed courses and seminars in customer service, sales strategies, inventory control, loss prevention, safety, time management, leadership and performance assessment. experience hr adm

In [9]:
job_description = """
We are looking for an HR Specialist with experience in employee relations,
recruitment, onboarding, and HR operations. The ideal candidate has strong
communication skills, experience with HRIS systems, and a background in
handling employee benefits and compliance.
"""

cleaned_jd = clean_text(job_description)
print(cleaned_jd)

we are looking for an hr specialist with experience in employee relations, recruitment, onboarding, and hr operations. the ideal candidate has strong communication skills, experience with hris systems, and a background in handling employee benefits and compliance.


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine job description + all resumes into one list
documents = [cleaned_jd] + df['cleaned_resume'].tolist()

# Convert text into TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(documents)

# Compare job description (first item) against every resume
similarity_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])

df['match_score'] = similarity_scores[0]
df[['Category', 'match_score']].sort_values(by='match_score', ascending=False).head(10)

,Category,match_score
31,HR,0.381217
4,HR,0.378243
90,HR,0.378068
68,HR,0.357275
75,HR,0.349200
85,HR,0.342282
18,HR,0.341829
65,HR,0.326876
58,HR,0.322632
9,HR,0.309367


In [11]:
def get_missing_keywords(jd_text, resume_text, top_n=15):
    # Fit TF-IDF on just these two documents
    vec = TfidfVectorizer(stop_words='english')
    tfidf = vec.fit_transform([jd_text, resume_text])
    
    feature_names = vec.get_feature_names_out()
    jd_scores = tfidf[0].toarray()[0]
    resume_scores = tfidf[1].toarray()[0]
    
    # Words that matter in the JD but are missing/weak in the resume
    missing = []
    for i, word in enumerate(feature_names):
        if jd_scores[i] > 0 and resume_scores[i] == 0:
            missing.append((word, jd_scores[i]))
    
    # Sort by importance in the JD, return top N
    missing.sort(key=lambda x: x[1], reverse=True)
    return [word for word, score in missing[:top_n]]

# Test it on the top-matching resume (index 31 from your results)
sample_resume = df.loc[31, 'cleaned_resume']
missing_keywords = get_missing_keywords(cleaned_jd, sample_resume)
print("Missing keywords:", missing_keywords)

Missing keywords: ['candidate', 'communication', 'handling', 'ideal', 'looking', 'specialist', 'strong']


In [12]:
# Try it on a low-scoring, unrelated resume - e.g. a CHEF resume
low_match_resume = df[df['Category'] == 'CHEF'].iloc[0]['cleaned_resume']
missing_keywords = get_missing_keywords(cleaned_jd, low_match_resume)
print("Missing keywords:", missing_keywords)

Missing keywords: ['employee', 'hr', 'background', 'benefits', 'compliance', 'handling', 'hris', 'ideal', 'looking', 'onboarding', 'recruitment', 'relations', 'specialist', 'systems']


In [13]:
# Common generic words that aren't real skills - filter these out
GENERIC_WORDS = {'ideal', 'looking', 'strong', 'candidate', 'background', 
                  'handling', 'specialist', 'experience', 'skills'}

def get_missing_keywords(jd_text, resume_text, top_n=15):
    vec = TfidfVectorizer(stop_words='english')
    tfidf = vec.fit_transform([jd_text, resume_text])
    
    feature_names = vec.get_feature_names_out()
    jd_scores = tfidf[0].toarray()[0]
    resume_scores = tfidf[1].toarray()[0]
    
    missing = []
    for i, word in enumerate(feature_names):
        if jd_scores[i] > 0 and resume_scores[i] == 0 and word not in GENERIC_WORDS:
            missing.append((word, jd_scores[i]))
    
    missing.sort(key=lambda x: x[1], reverse=True)
    return [word for word, score in missing[:top_n]]

# Re-test on resume #31 (our top match)
sample_resume = df.loc[31, 'cleaned_resume']
print("Top match missing keywords:", get_missing_keywords(cleaned_jd, sample_resume))

Top match missing keywords: ['communication']


In [15]:
from sklearn.model_selection import train_test_split

X = df['cleaned_resume']
y = df['Category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1987
Testing samples: 497


In [16]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(1987, 5000)
(497, 5000)


In [17]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

print("Model trained!")

Model trained!


In [18]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

print("\nDetailed report:")
print(classification_report(y_test, y_pred))

Accuracy: 64.99%

Detailed report:
                        precision    recall  f1-score   support

            ACCOUNTANT       0.67      0.83      0.74        24
              ADVOCATE       0.35      0.54      0.43        24
           AGRICULTURE       1.00      0.46      0.63        13
               APPAREL       0.67      0.21      0.32        19
                  ARTS       0.46      0.29      0.35        21
            AUTOMOBILE       0.00      0.00      0.00         7
              AVIATION       0.85      0.71      0.77        24
               BANKING       0.83      0.65      0.73        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.46      0.79      0.58        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.78      0.82      0.80        22
            CONSULTANT       0.44      0.17      0.25        23
              DESIGNER       0.84      0.76      0.80        21
    

c:\Users\absho\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\absho\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\absho\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave